# PySpark Intermediate Level Transformations

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/introduction/raw-data/BigMart Sales.csv")
)

In [0]:
display(df.limit(11))

---

## DROP DUPLICATE
dropDuplicates() digunakan untuk menghapus baris yang duplikat pada DataFrame PySpark.\
Konsepnya sama seperti drop_duplicates() di Pandas atau SELECT DISTINCT di SQL.

In [0]:
df_dropdup = df.dropDuplicates()
df_dropdup.count()

In [0]:
df_dropdup = (
    df
    .orderBy(col("Item_Outlet_Sales").desc())
    .drop_duplicates(subset=["Item_Type"])
)

display(df_dropdup)

---

## UNION & UNION BY NAME

In [0]:
# Preparing the Dataframe

data1 = [('1', 'kad'), 
          ('2', 'sid')]          
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)

data2 = [('3', 'rahul'), 
         ('4','jas')]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2, schema2)

In [0]:
df_union = df1.union(df2)
display(df_union)

In [0]:
df_union_auto = df1.unionByName(df2)
display(df_union_auto)

---

## STRING FUNCTIONS

In [0]:
"""
Sebagian besar string functions digunakan di dalam withColumn().

String functions di PySpark berada di pyspark.sql.functions.
Operasi dilakukan pada kolom, bukan pada objek string Python.
Fungsi-fungsi ini biasanya digunakan di dalam withColumn(), select(), atau filter().
Fungsi yang paling sering digunakan dalam ETL adalah upper, lower, trim, length, concat, concat_ws, substring, split, regexp_replace, contains, startswith, dan endswith.
Berbeda dengan Pandas yang menggunakan .str, PySpark menggunakan fungsi-fungsi SQL yang dapat dioptimalkan oleh Spark dan dieksekusi secara terdistribusi di seluruh cluster

INGAT: col("column_name") itu seperti df["column_name"]
Operasinya dilakukan secara VECTORIZED (otomatis tanpa loop)
TANPA menyebut col() juga sebenarnya bisa.. col() essensial untuk METHOD
"""

In [0]:
df_upper_subset = df.select(upper(col("Item_Type")).alias("item_type"))
display(df_upper_subset)

In [0]:
df_upper = df.withColumn("Item_Type", upper("Item_Type"))
display(df_upper.limit(7))

---

## DATE FUNCTIONS

In [0]:
"""
Date functions di PySpark digunakan untuk membaca, mengubah, menghitung, dan memanipulasi data bertipe Date atau Timestamp.

| Tipe          | Contoh              |
| ------------- | ------------------- |
| DateType      | 2026-07-03          |
| TimestampType | 2026-07-03 15:30:45 |

| Operasi        | Pandas                | PySpark         |
| -------------- | --------------------- | --------------- |
| Tahun          | `.dt.year`            | `year()`        |
| Bulan          | `.dt.month`           | `month()`       |
| Hari           | `.dt.day`             | `dayofmonth()`  |
| Tambah hari    | `+ pd.Timedelta()`    | `date_add()`    |
| Kurangi hari   | `- pd.Timedelta()`    | `date_sub()`    |
| Selisih hari   | `(end-start).dt.days` | `datediff()`    |
| Format tanggal | `.dt.strftime()`      | `date_format()` |

- Gunakan to_date() atau to_timestamp() untuk mengonversi string menjadi tipe tanggal atau waktu sebelum melakukan operasi tanggal.
- Gunakan year(), month(), dayofmonth(), weekofyear(), dan fungsi serupa untuk mengekstrak komponen tanggal.
- Gunakan date_add(), date_sub(), add_months(), dan datediff() untuk perhitungan tanggal.
- Gunakan current_date() dan current_timestamp() untuk mendapatkan tanggal atau waktu saat eksekusi.
- Gunakan date_format() hanya jika membutuhkan representasi string, misalnya untuk laporan atau ekspor data, karena hasilnya bukan lagi bertipe DateType
"""

### Current Date

In [0]:
df_date = df.withColumn("Current_Date", current_date())
display(df_date.limit(3))

### Date Add & Date Sub

In [0]:
df_date = (
    df_date.withColumn("Week_After", date_add(col("Current_Date"), 7))
    .withColumn("Week_Before", date_sub(col("Current_Date"), 7)) # date_add(col, -7)
)

display(df_date.limit(5))

### Date Diff

In [0]:
df_date = df_date.withColumn("Date_Diff", datediff("Week_After", "Week_Before"))
display(df_date.limit(3))

### Date Format

In [0]:
df_date = df_date.withColumn("Current_Date", date_format(col("Current_Date"), "dd/MM/yyyy"))
display(df_date.limit(3))

---

## HANDLING NULLS


In [0]:
# Jika menggunakan withColumn maka kita umumnya menggunakan FUNCTION
# Jika menggunakan method maka langsung diterapkan di df nya
# Pokoknya gunakan withColumn jika mau menambah column baru atau mengubah value existing column
# Perubahan withCOlumn itu level COLUMN bukan DATAFRAME

In [0]:
display(df.limit(10))

### Drop Nulls

In [0]:
# Hapus row jika seluruh column pada row tersebut bernilai Nulls
df_nulls = df.dropna("all")
df_nulls.count()

In [0]:
# Hapus suatu row jika ada minimal SATU nulls pada suatu nilai pada row tsb
df_nulls = df.dropna("any")
df_nulls.count()

In [0]:
# Hapus row jika ada nilai nulls pada subset/column tertentu (recommended)
df_nulls = df.dropna(subset=["Outlet_Size"])
df_nulls.count()

### Fill Nulls

In [0]:
# Filling ALL the Nulls value in the dataframe
df_fillna = df.fillna("NotAvailable") # Karena nilainya adalah string maka HANYA akan memfill nulls pada STRING
display(df_fillna.limit(9))

In [0]:
# Fill nulls value pada column tertentu (subset)
df_fillna = df.fillna(70, subset=["Item_Weight"])
display(df_fillna.limit(9))

---

## SPLIT & INDEXING

### Split

In [0]:
df_split = df.withColumn("Outlet_Type", split("Outlet_Type", " "))
display(df_split.limit(3))

In [0]:
df_split = df.withColumn("Outlet_Type", split("Outlet_Type", " ")[0])
display(df_split.limit(3))

In [0]:
df_split = df.withColumn("Outlet_Type", split("Outlet_Type", " "))

df_split = (
    df_split.withColumn("Outlet", col("Outlet_Type")[0])
    .withColumn("Type", col("Outlet_Type")[1])
    .drop("Outlet_Type")
)

display(df_split.limit(5))

### Explode

In [0]:
# Explode seperti memflattenkan json/ mengunpack list jadi masing2 rows
df_split = df.withColumn("Outlet_Type", split("Outlet_Type", " "))
df_explode = df_split.withColumn("Outlet_Type", explode("Outlet_Type"))
display(df_explode.limit(5))

### Array Contains

In [0]:
display(df_split.limit(3))

In [0]:
df_arcon = df_split.withColumn("Flag_Outlet", array_contains("Outlet_Type", "Type1"))
display(df_arcon.limit(5))

---

## Group By

In [0]:
"""
groupBy() di PySpark digunakan untuk mengelompokkan data berdasarkan satu atau lebih kolom, biasanya diikuti dengan fungsi agregasi seperti sum(), count(), avg(), max(), atau min().

- groupBy() mengelompokkan data berdasarkan satu atau lebih kolom.
- Hasil groupBy() adalah objek GroupedData, bukan DataFrame.
- Gunakan agg() untuk melakukan satu atau beberapa agregasi seperti sum, avg, count, min, dan max.
- groupBy() di PySpark memiliki konsep yang sama dengan GROUP BY di SQL.
- Dalam praktik Data Engineering, pola yang paling umum adalah groupBy(...).agg(...) dengan alias() untuk memberi nama hasil agregasi yang jelas.
"""

In [0]:
display(df.limit(7))

In [0]:
df_agg = df.groupBy("Item_Type").sum("Item_Outlet_Sales")
display(df_agg)

In [0]:
df_multi_agg = df.groupBy("Item_Type").agg(sum("Item_Outlet_Sales").alias("Total_Sales"), avg("Item_Outlet_Sales").alias("Avg_Sales"))

display(df_multi_agg)

In [0]:
df_agg = df.groupBy("Item_Type", "Outlet_Size")\
           .agg(sum("Item_Outlet_Sales").alias("Total_Sales"))\
           .orderBy(col("Total_Sales").desc())
display(df_agg)